In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import scipy.io as scio
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, roc_auc_score, recall_score,
    precision_score, f1_score, confusion_matrix,
    roc_curve, auc
)
from sklearn.preprocessing import StandardScaler
import pandas as pd
from tqdm import tqdm
import pickle
from datetime import datetime
import random
import gc
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from scipy import stats
from scipy.stats import gaussian_kde
import nibabel as nib
from nilearn.plotting import plot_prob_atlas
from nilearn import image
from nilearn.image import new_img_like

print("✅ All packages loaded successfully!")


In [ ]:
# STYLE SETUP FOR VISUALIZATIONS
# ============================================================================
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except OSError:
    try:
        plt.style.use('seaborn-darkgrid')
    except OSError:
        plt.style.use('default')
        plt.rcParams['grid.alpha'] = 0.3
        plt.rcParams['grid.linestyle'] = '--'
        plt.rcParams['axes.facecolor'] = '#f0f0f0'
        plt.rcParams['figure.facecolor'] = 'white'

sns.set_palette("husl")

# ============================================================================
# FOCAL LOSS - For class imbalance
# ============================================================================

class FocalLoss(nn.Module):
    """Focal loss for handling class imbalance"""
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets,
                             weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()



In [ ]:
# ============================================================================
# 2. PREPROCESSING
# ============================================================================
def upper_tri_features(C):
    """(N, 200, 200) -> (N, 19900) upper-triangular edge features."""
    iu = np.triu_indices(N_ROIS, k=1)
    return C[:, iu[0], iu[1]]


def features_to_matrices(X):
    """Inverse of upper_tri_features, with unit diagonal."""
    iu = np.triu_indices(N_ROIS, k=1)
    C = np.zeros((X.shape[0], N_ROIS, N_ROIS), dtype=np.float32)
    C[:, iu[0], iu[1]] = X
    C = C + np.transpose(C, (0, 2, 1))
    idx = np.arange(N_ROIS)
    C[:, idx, idx] = 1.0
    return C


def zscore_per_subject(C, eps=1e-8):
    """Eq. (1): per-subject z-score of the connectivity matrix."""
    mu = C.mean(axis=(1, 2), keepdims=True)
    sd = C.std(axis=(1, 2), keepdims=True)
    return (C - mu) / (sd + eps)


def build_normalised_adjacency(C, top_pct=90):
    """
    Eqs. (3)-(7): threshold |C| at its top_pct percentile per subject,
    add self-loops, and apply symmetric degree normalisation.
    """
    absC = np.abs(C)
    iu = np.triu_indices(N_ROIS, k=1)
    thr = np.percentile(absC[:, iu[0], iu[1]], top_pct, axis=1)     # (N,)
    A = (absC > thr[:, None, None]).astype(np.float32)               # Eq. (4)
    idx = np.arange(N_ROIS)
    A[:, idx, idx] = 0.0
    A = A + np.eye(N_ROIS, dtype=np.float32)                         # Eq. (5)
    deg = A.sum(axis=2)                                              # Eq. (7)
    d_inv_sqrt = 1.0 / np.sqrt(deg)
    A_hat = d_inv_sqrt[:, :, None] * A * d_inv_sqrt[:, None, :]      # D^-1/2 A' D^-1/2
    return A_hat.astype(np.float32)


class ComBatHarmoniser:
    """
    Location-scale ComBat (Johnson et al., 2007; Fortin et al., 2018) without
    empirical-Bayes shrinkage. Parameters are estimated on the fitting
    participants only and then applied to any participant. Covariates
    (age, sex) are preserved; diagnosis is not a covariate.
    """

    def fit(self, Y, batch, covars):
        Y = np.asarray(Y, dtype=np.float64)
        self.levels = np.unique(batch)
        K = len(self.levels)
        B = (batch[:, None] == self.levels[None, :]).astype(np.float64)
        X = np.hstack([B, covars])
        coef = np.linalg.lstsq(X, Y, rcond=None)[0]

        n = Y.shape[0]
        n_b = B.sum(axis=0)
        self.alpha = (n_b[:, None] / n * coef[:K]).sum(axis=0)     # grand mean
        self.beta = coef[K:]                                       # covariate effects
        resid = Y - X @ coef
        self.sigma = np.sqrt((resid ** 2).sum(axis=0) / max(n - X.shape[1], 1)) + 1e-8

        Z = (Y - self.alpha - covars @ self.beta) / self.sigma
        self.gamma = np.zeros((K, Y.shape[1]))
        self.delta = np.ones((K, Y.shape[1]))
        for k, lev in enumerate(self.levels):
            m = batch == lev
            self.gamma[k] = Z[m].mean(axis=0)
            if m.sum() > 1:
                self.delta[k] = Z[m].var(axis=0, ddof=1)
        self.delta = np.maximum(self.delta, 1e-6)
        return self

    def transform(self, Y, batch, covars):
        Y = np.asarray(Y, dtype=np.float64)
        if not np.isin(batch, self.levels).all():
            raise ValueError("ComBat: a site is absent from the fitting (training) fold")
        cov_eff = covars @ self.beta
        Z = (Y - self.alpha - cov_eff) / self.sigma
        out = np.empty_like(Z)
        for k, lev in enumerate(self.levels):
            m = batch == lev
            out[m] = (Z[m] - self.gamma[k]) / np.sqrt(self.delta[k])
        return out * self.sigma + self.alpha + cov_eff


def prepare_fold(data, fit_idx, cfg):
    """
    Build model inputs for one fold. Every statistic (ComBat parameters, age
    z-score, feature mask, sMRI standardisation) is estimated on fit_idx only.
    """
    site = data["site"]
    age, sex = data["age"], data["sex"]
    age_mu, age_sd = age[fit_idx].mean(), age[fit_idx].std() + 1e-8
    covars = np.column_stack([(age - age_mu) / age_sd, sex]).astype(np.float64)

    # ---- fMRI: ComBat on FC edges -> z-scored node features + graph
    fc_feat = data["fc_feat"]
    if cfg["use_combat"]:
        cb = ComBatHarmoniser().fit(fc_feat[fit_idx], site[fit_idx], covars[fit_idx])
        fc_feat = cb.transform(fc_feat, site, covars)
    fc = features_to_matrices(fc_feat)
    nodes = zscore_per_subject(fc)                         # Eq. (1)
    adj = build_normalised_adjacency(fc, cfg["top_pct"])   # Eqs. (3)-(7)

    # ---- sMRI: drop constant features, ComBat, standardise (Eq. 2)
    smri = data["smri"]
    keep = smri[fit_idx].std(axis=0) > 1e-8
    smri = smri[:, keep]
    if cfg["use_combat"]:
        cb_s = ComBatHarmoniser().fit(smri[fit_idx], site[fit_idx], covars[fit_idx])
        smri = cb_s.transform(smri, site, covars)
    mu = smri[fit_idx].mean(axis=0)
    sd = smri[fit_idx].std(axis=0) + 1e-8
    smri = np.nan_to_num((smri - mu) / sd)                 # Eq. (2)

    return nodes.astype(np.float32), adj, smri.astype(np.float32)

In [ ]:
# ============================================================================

# ============================================================================
# 1. GRAPH CONVOLUTIONAL LAYERS - fMRI GCN
# ============================================================================

class GCNLayer(nn.Module):
    """Graph Convolutional Layer"""
    def __init__(self, in_dim, out_dim, dropout=0.3):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim, bias=False)
        self.bn     = nn.BatchNorm1d(out_dim)
        self.drop   = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.linear.weight)

    def forward(self, H, A_hat):
        out = A_hat @ H
        out = self.linear(out)
        out = self.bn(out)
        out = F.relu(out)
        return self.drop(out)


class fMRI_GCN(nn.Module):
    """fMRI Processing: 3-layer GCN + pooling"""
    def __init__(self, n_rois=200, gcn_dims=(200, 256, 128),
                 out_dim=128, dropout=0.3):
        super().__init__()
        self.n_rois = n_rois
        self.out_dim = out_dim

        layers = []
        in_d = gcn_dims[0]
        for out_d in gcn_dims[1:]:
            layers.append(GCNLayer(in_d, out_d, dropout))
            in_d = out_d
        self.gcn_layers = nn.ModuleList(layers)

        pool_dim = gcn_dims[-1] * 2

        self.mlp = nn.Sequential(
            nn.Linear(pool_dim, out_dim * 2),
            nn.BatchNorm1d(out_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(out_dim * 2, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(),
        )
        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    @staticmethod
    def build_normalised_adj(conn: np.ndarray, keep_pct=20) -> torch.Tensor:
        """Build normalized adjacency matrix from correlation matrix"""
        N = conn.shape[0]
        
        conn = np.nan_to_num(conn, nan=0.0, posinf=0.0, neginf=0.0)
        
        thr = np.percentile(np.abs(conn), 100 - keep_pct)
        A = np.where(np.abs(conn) >= thr, np.abs(conn), 0.0).astype(np.float32)
        np.fill_diagonal(A, 1.0)
        
        deg = A.sum(1)
        deg_inv_sq = np.where(deg > 0, deg ** -0.5, 0.0)
        A_hat = deg_inv_sq[:, None] * A * deg_inv_sq[None, :]
        
        A_hat = np.nan_to_num(A_hat, nan=0.0, posinf=0.0, neginf=0.0)
        
        return torch.from_numpy(A_hat.astype(np.float32))

    def forward(self, conn_batch: torch.Tensor) -> torch.Tensor:
        """Forward pass"""
        B, N, _ = conn_batch.shape
        device = conn_batch.device
        all_emb = []

        for i in range(B):
            try:
                conn_np = conn_batch[i].cpu().float().numpy()
                H = torch.from_numpy(conn_np).float().to(device)
                A_hat = self.build_normalised_adj(conn_np).to(device)

                x = H
                for layer in self.gcn_layers:
                    x = layer(x, A_hat)

                mean_p = x.mean(0)
                max_p = x.max(0).values
                emb = torch.cat([mean_p, max_p])
                all_emb.append(emb)
            except Exception as e:
                print(f"⚠️ GCN error at batch {i}: {e}")
                all_emb.append(torch.zeros(self.out_dim * 2, device=device))

        stacked = torch.stack(all_emb)
        return self.mlp(stacked)


In [ ]:
# ============================================================================
# 2. STRUCTURAL MRI MLP - sMRI Processing
# ============================================================================

class sMRI_MLP(nn.Module):
    """sMRI Processing: multi-scale MLP.
    The stem embeds the sMRI vector into D = 2*channels features. Each scale k
    average-pools those features in contiguous groups of size k, runs its own
    MLP branch, and the branch outputs are fused."""
    def __init__(self, in_features=1417, channels=128,
                 out_dim=128, dropout=0.3, scales=(1, 2, 4)):
        super().__init__()
        D = channels * 2
        self.scales = scales

        self.input_embed = nn.Sequential(
            nn.Linear(in_features, channels * 4),
            nn.BatchNorm1d(channels * 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(channels * 4, D),
            nn.BatchNorm1d(D),
            nn.ReLU(),
        )

        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Linear(D // k, channels),
                nn.BatchNorm1d(channels),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(channels, channels),
                nn.BatchNorm1d(channels),
                nn.ReLU(),
            )
            for k in scales
        ])

        self.fuse = nn.Sequential(
            nn.Linear(channels * len(scales), channels),
            nn.BatchNorm1d(channels),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.proj = nn.Sequential(
            nn.Linear(channels, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(),
        )
        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_embed(x.float())
        B = x.size(0)
        outs = []
        for k, branch in zip(self.scales, self.branches):
            xs = x if k == 1 else x.reshape(B, -1, k).mean(dim=-1)
            outs.append(branch(xs))
        fused = self.fuse(torch.cat(outs, dim=1))
        return self.proj(fused)



In [ ]:
# ============================================================================
# 3. MULTI-SCALE ATTENTION FUSION WITH DIVERSITY ENFORCEMENT
# ============================================================================

class MultiScaleAttentionFusion(nn.Module):
    """Multi-Scale Attention Fusion with gate diversity enforcement"""
    def __init__(self, hidden_dim=128, num_heads=4, dropout=0.3):
        super().__init__()
        H = hidden_dim
        self.H = H
        self.num_heads = num_heads

        # Scale 1: Full dimension
        self.ln_pre1 = nn.LayerNorm(H)
        self.mha1 = nn.MultiheadAttention(
            H, num_heads, dropout=dropout, batch_first=True)
        self.ln1 = nn.LayerNorm(H)
        self.ff1 = nn.Sequential(
            nn.Linear(H, H * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(H * 2, H))
        self.ln1b = nn.LayerNorm(H)

        # Scale 2: Half dimension
        H2 = max(16, H // 2)
        self.H2 = H2
        self.dn2 = nn.Linear(H, H2, bias=False)
        self.ln_pre2 = nn.LayerNorm(H2)
        self.mha2 = nn.MultiheadAttention(
            H2, max(1, num_heads // 2), dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(H2)
        self.up2 = nn.Linear(H2, H, bias=False)

        # Scale 3: Quarter dimension
        H4 = max(8, H // 4)
        self.H4 = H4
        self.dn3 = nn.Linear(H, H4, bias=False)
        self.ln_pre3 = nn.LayerNorm(H4)
        self.mha3 = nn.MultiheadAttention(
            H4, max(1, num_heads // 4), dropout=dropout, batch_first=True)
        self.ln3 = nn.LayerNorm(H4)
        self.up3 = nn.Linear(H4, H, bias=False)

        # Gate weighting
        self.gate_net = nn.Sequential(
            nn.Linear(H * 3, H),
            nn.LayerNorm(H),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(H, 3),
            nn.Sigmoid()
        )

        self.out_proj = nn.Sequential(
            nn.Linear(H, H),
            nn.LayerNorm(H),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(H, H),
            nn.LayerNorm(H),
        )

        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
        nn.init.zeros_(self.gate_net[-2].weight)
        nn.init.constant_(self.gate_net[-2].bias, 0.0)

    def _pool2(self, x):
        return x.mean(dim=1)

    def forward(self, tok_fmri, tok_smri):
        """Forward pass"""
        tok = torch.stack([tok_fmri, tok_smri], dim=1)

        # Scale 1: Full dimension
        t1 = self.ln_pre1(tok)
        s1, _ = self.mha1(t1, t1, t1)
        s1 = self.ln1(s1 + tok)
        s1 = self.ln1b(self.ff1(s1) + s1)
        emb1 = self._pool2(s1)

        # Scale 2: Half dimension
        tok2 = self.dn2(tok)
        t2 = self.ln_pre2(tok2)
        s2, _ = self.mha2(t2, t2, t2)
        s2 = self.ln2(s2 + tok2)
        emb2 = self.up2(self._pool2(s2))

        # Scale 3: Quarter dimension
        tok3 = self.dn3(tok)
        t3 = self.ln_pre3(tok3)
        s3, _ = self.mha3(t3, t3, t3)
        s3 = self.ln3(s3 + tok3)
        emb3 = self.up3(self._pool2(s3))

        # Gate weighting
        gate_in = torch.cat([emb1, emb2, emb3], dim=1)
        gate_w = self.gate_net(gate_in)

        fused = (gate_w[:, 0:1] * emb1 +
                 gate_w[:, 1:2] * emb2 +
                 gate_w[:, 2:3] * emb3)

        # Diversity loss
        uniform = torch.ones_like(gate_w) / 3.0
        div_loss = ((gate_w - uniform) ** 2).sum(dim=1).mean()

        eps = 1e-8
        entropy = -(gate_w * torch.log(gate_w + eps)).sum(dim=1).mean()
        
        div_loss = div_loss - 0.1 * entropy

        return self.out_proj(fused), gate_w, div_loss



In [ ]:
# ============================================================================
# 4. COMPLETE HYBRID MODEL
# ============================================================================

class HybridGCN_MLP_MSA(nn.Module):
    """Complete model with all components"""
    def __init__(self, smri_features=1417, hidden_dim=128,
                 gcn_hidden=128, num_heads=4, dropout=0.3, n_rois=200):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.fmri_gcn = fMRI_GCN(
            n_rois=n_rois,
            gcn_dims=(n_rois, gcn_hidden * 2, gcn_hidden),
            out_dim=hidden_dim,
            dropout=dropout
        )

        self.smri_mlp = sMRI_MLP(
            in_features=smri_features,
            channels=hidden_dim,
            out_dim=hidden_dim,
            dropout=dropout
        )

        self.fusion = MultiScaleAttentionFusion(
            hidden_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 2)
        )

        nn.init.normal_(self.classifier[-1].weight, std=0.01)
        nn.init.zeros_(self.classifier[-1].bias)

    def forward(self, conn, smri):
        """Forward pass"""
        tok_f = self.fmri_gcn(conn)
        tok_s = self.smri_mlp(smri.float())
        fused, gate_w, div_loss = self.fusion(tok_f, tok_s)
        logits = self.classifier(fused)

        extras = {
            'tok_fmri': tok_f.detach(),
            'tok_smri': tok_s.detach(),
            'gate_w': gate_w.detach(),
            'div_loss': div_loss,
        }

        return logits, extras



In [ ]:
# ============================================================================
# 5. DATASET CLASS
# ============================================================================

class BrainDataset(Dataset):
    """PyTorch Dataset for brain imaging"""
    def __init__(self, conn, smri, labels):
        self.conn = conn.astype(np.float32)
        self.smri = smri.astype(np.float32)
        self.labels = labels.astype(np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        conn = torch.from_numpy(self.conn[idx]).float()
        smri = torch.from_numpy(self.smri[idx]).float()
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return conn, smri, label


def make_sampler(labels):
    """Weighted sampler for class imbalance"""
    cnt = np.bincount(labels)
    w = 1.0 / cnt[labels]
    return WeightedRandomSampler(
        torch.from_numpy(w).float(), len(labels), replacement=True)



In [ ]:
# ============================================================================
# 6. DATA LOADING
# ============================================================================

def load_all_data(base_path):
    """Load ABIDE data - fMRI connectivity + sMRI features"""
    print("📂 Loading ABIDE data…")

    fmri_p = os.path.join(base_path, "Path")
    smri_p = os.path.join(base_path, "Path")
    lbl_p = os.path.join(base_path, "Path")

    # Load metadata
    sids = np.genfromtxt(os.path.join(lbl_p, 'subject_IDs.txt'), dtype=str)
    lbls = scio.loadmat(
        os.path.join(lbl_p, 'ABIDE_label_870.mat'))['label'][0]
    sr = scio.loadmat(
        os.path.join(lbl_p, 'sites.mat'))['sites'].flatten()
    sites = np.array([str(s).replace(' ', '').replace("'", '') for s in sr])

    print(f"  Total subjects: {len(sids)}")
    print(f"  ASD={np.sum(lbls == 1)}, Control={np.sum(lbls == 0)}")

    # Load fMRI connectivity
    cl, vl, vs = [], [], []
    for i, sid in enumerate(tqdm(sids, desc='Loading fMRI')):
        if sid == '51334':
            continue
        fp = os.path.join(fmri_p, f"{sid}.mat")
        if not os.path.exists(fp):
            continue
        try:
            d = scio.loadmat(fp)
            if 'connectivity' not in d:
                continue
            c = d['connectivity'].astype(np.float32)
            
            if c.shape != (200, 200):
                if c.shape[0] == c.shape[1]:
                    m = min(c.shape[0], 200)
                    t = np.zeros((200, 200), np.float32)
                    t[:m, :m] = c[:m, :m]
                    c = t
                else:
                    continue
            
            if np.any(np.isnan(c)) or np.any(np.isinf(c)):
                continue
            
            cl.append(c)
            vl.append(int(lbls[i]))
            vs.append(sites[i])
        except:
            continue

    if len(cl) == 0:
        print("❌ No fMRI data loaded")
        return None

    conn = np.array(cl, np.float32)
    vl = np.array(vl, np.int64)
    vs = np.array(vs)
    print(f"✅ Loaded fMRI: {conn.shape}")

    # Load sMRI features
    smri_features = None
    npz = os.path.join(smri_p, "with_phe", "all_modalities_combined.npz")
    
    if os.path.exists(npz):
        try:
            d = np.load(npz, allow_pickle=True)
            if 'smri_features' in d:
                s = d['smri_features'].astype(np.float32)
                n = len(conn)
                if s.shape[0] >= n:
                    smri_features = s[:n]
                else:
                    padded = np.zeros((n, s.shape[1]), np.float32)
                    padded[:s.shape[0]] = s
                    smri_features = padded
        except Exception as e:
            print(f"⚠️ sMRI loading error: {e}")

    if smri_features is None:
        print("❌ sMRI features not found")
        return None

    smri_features = np.nan_to_num(smri_features, nan=0.0, posinf=0.0, neginf=0.0)
    print(f"✅ Loaded sMRI: {smri_features.shape}")

    print(f"\n✅ FINAL DATASET:")
    print(f"   Samples: {len(conn)}")
    print(f"   fMRI: {conn.shape}")
    print(f"   sMRI: {smri_features.shape}")
    print(f"   ASD: {np.sum(vl)}, Control: {np.sum(vl == 0)}")
    print(f"   Sites: {len(np.unique(vs))}")

    return {
        'connectivity': conn,
        'smri_features': smri_features,
        'labels': vl,
        'sites': vs
    }


In [ ]:
# ============================================================================
# 7. UTILITY FUNCTIONS
# ============================================================================

def set_seed(s=42):
    """Set all random seeds"""
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def norm_conn_zscore(conn):
    """Z-score normalize connectivity per sample"""
    out = np.zeros_like(conn, np.float32)
    for i in range(len(conn)):
        mu = conn[i].mean()
        sd = conn[i].std() + 1e-8
        out[i] = (conn[i] - mu) / sd
    return out


def class_weights_tensor(labels, device):
    """Compute class weights for focal loss"""
    cnt = np.bincount(labels)
    w = len(labels) / (len(cnt) * cnt)
    return torch.tensor(w, dtype=torch.float32, device=device)


def make_folds(sites, labels, n=5, seed=42):
    """Create stratified k-fold with validation split"""
    skf = StratifiedGroupKFold(n_splits=n, shuffle=True, random_state=seed)
    folds = []
    
    for fi, (tv, te) in enumerate(
            skf.split(np.arange(len(labels)), y=labels, groups=sites)):
        
        inner = StratifiedGroupKFold(
            n_splits=4, shuffle=True, random_state=seed + fi)
        
        tr_r, va_r = next(iter(inner.split(
            np.arange(len(tv)), y=labels[tv], groups=sites[tv])))
        
        folds.append({
            'train': tv[tr_r],
            'val': tv[va_r],
            'test': te
        })
    
    return folds


def calc_metrics(yt, yp, ypr):
    """Calculate all metrics"""
    if len(np.unique(yt)) < 2:
        return None
    
    try:
        tn, fp, fn, tp = confusion_matrix(yt, yp).ravel()
        return dict(
            accuracy=accuracy_score(yt, yp),
            sensitivity=recall_score(yt, yp, zero_division=0),
            specificity=recall_score(yt, yp, pos_label=0, zero_division=0),
            precision=precision_score(yt, yp, zero_division=0),
            f1_score=f1_score(yt, yp, zero_division=0),
            auc=roc_auc_score(yt, ypr),
            tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp)
        )
    except Exception as e:
        print(f"⚠️ Metrics error: {e}")
        return None


def cosine_warmup(opt, warmup, total):
    """Cosine annealing with linear warmup"""
    from torch.optim.lr_scheduler import LambdaLR
    fn = lambda ep: (ep / max(1, warmup) if ep < warmup else
                     max(0.0, (total - ep) / max(1, total - warmup)))
    return LambdaLR(opt, fn)



In [ ]:
# ============================================================================
# 8. TRAINING LOOP
# ============================================================================

def run_epoch(model, loader, device, criterion,
              optimizer=None, div_weight=0.2,
              mixup_alpha=0.2, is_train=True):
    """Run one training/validation epoch"""
    if is_train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    preds_all, probs_all, trues_all = [], [], []
    gate_all = []

    ctx = torch.enable_grad() if is_train else torch.no_grad()

    with ctx:
        for conn_b, smri_b, lbl_b in loader:
            conn_b = conn_b.to(device).float()
            smri_b = smri_b.to(device).float()
            lbl_b = lbl_b.to(device)

            if is_train:
                if mixup_alpha > 0:
                    lam = np.random.beta(mixup_alpha, mixup_alpha)
                    idx = torch.randperm(conn_b.shape[0], device=device)
                    conn_m = lam * conn_b + (1 - lam) * conn_b[idx]
                    smri_m = lam * smri_b + (1 - lam) * smri_b[idx]
                    lbl_a, lbl_b2 = lbl_b, lbl_b[idx]
                else:
                    conn_m, smri_m = conn_b, smri_b
                    lbl_a = lbl_b2 = lbl_b
                    lam = 1.0

                optimizer.zero_grad()

            try:
                if is_train:
                    logits, extras = model(conn_m, smri_m)
                    task_loss = (lam * criterion(logits, lbl_a) +
                                 (1 - lam) * criterion(logits, lbl_b2))
                else:
                    logits, extras = model(conn_b, smri_b)
                    task_loss = criterion(logits, lbl_b)

                div_loss = extras['div_loss']
                loss = task_loss + div_weight * div_loss

                if is_train:
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()

                with torch.no_grad():
                    prob = torch.softmax(logits, 1)
                    preds_all.extend(logits.argmax(1).cpu().numpy())
                    probs_all.extend(prob[:, 1].cpu().numpy())
                    trues_all.extend(lbl_b.cpu().numpy())
                    gate_all.append(extras['gate_w'].cpu().numpy())
                    total_loss += task_loss.item() * lbl_b.shape[0]

            except Exception as e:
                print(f"  ⚠️ Batch error: {e}")
            finally:
                if is_train:
                    del conn_b, smri_b, lbl_b
                torch.cuda.empty_cache()

    n = len(trues_all)
    acc = (np.array(preds_all) == np.array(trues_all)).mean() if n else 0.0
    loss = total_loss / max(n, 1)

    try:
        auc = roc_auc_score(np.array(trues_all), np.array(probs_all)) \
              if len(np.unique(trues_all)) > 1 else 0.5
    except:
        auc = 0.5

    gw = np.concatenate(gate_all).mean(0) if gate_all else np.zeros(3)

    return dict(loss=loss, acc=acc, auc=auc,
                preds=np.array(preds_all),
                probs=np.array(probs_all),
                trues=np.array(trues_all),
                gate_w=gw)


def train_model(data_dict, device, n_splits=5, config=None):
    """Complete 5-fold cross-validation training"""
    print(f"\n{'='*80}")
    print("🚀 TRAINING: fMRI-GCN + sMRI-MLP + Multi-Scale Attention")
    print("   WITH GATE DIVERSITY ENFORCEMENT")
    print(f"{'='*80}\n")

    if config is None:
        config = dict(
            epochs=100,
            batch_size=16,
            lr=3e-4,
            dropout=0.3,
            hidden_dim=128,
            gcn_hidden=128,
            num_heads=4,
            patience=25,
            min_epochs=20,
            weight_decay=1e-4,
            focal_gamma=2.0,
            mixup_alpha=0.2,
            div_weight=0.2,
            balanced_sampler=True,
        )

    print("Configuration:")
    for key, val in config.items():
        print(f"  {key:20s}: {val}")

    print(f"\n📊 Creating {n_splits}-fold cross-validation…")
    folds = make_folds(data_dict['sites'], data_dict['labels'], n_splits)
    print(f"✅ Created {len(folds)} folds")

    all_results = []
    all_logs = []


In [ ]:
 # ====================================================================
        # Create model
        # ====================================================================
        model = HybridGCN_MLP_MSA(
            smri_features=smri_train.shape[1],
            hidden_dim=config['hidden_dim'],
            gcn_hidden=config['gcn_hidden'],
            num_heads=config['num_heads'],
            dropout=config['dropout'],
            n_rois=200,
        ).to(device)

        params = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"\n📈 Model:")
        print(f"   Total parameters: {params:,}")
        print(f"   Trainable: {trainable:,}")

        # ====================================================================
        # Loss and optimizer
        # ====================================================================
        cw = class_weights_tensor(l_tr, device)
        criterion = FocalLoss(alpha=cw, gamma=config['focal_gamma'])
        print(f"   Class weights: ASD={cw[1]:.3f}, Control={cw[0]:.3f}")

        optimizer = optim.AdamW(
            model.parameters(),
            lr=config['lr'],
            weight_decay=config['weight_decay']
        )

        scheduler = cosine_warmup(
            optimizer,
            warmup=5,
            total=config['epochs']
        )

        best_val_auc = 0.0
        patience = 0
        best_state = None
        epoch_log = []

In [ ]:
# ====================================================================
        # TRAINING EPOCHS
        # ====================================================================
        print("\n🏋️  Training…")
        
        for epoch in range(config['epochs']):
            # Train
            tr_res = run_epoch(
                model, train_loader, device, criterion,
                optimizer=optimizer,
                div_weight=config['div_weight'],
                mixup_alpha=config['mixup_alpha'],
                is_train=True
            )

            # Validate
            model.eval()
            va_res = run_epoch(
                model, val_loader, device, criterion,
                div_weight=config['div_weight'],
                is_train=False
            )
            model.train()

            scheduler.step()

            # Log
            epoch_log.append(dict(
                epoch=epoch + 1,
                tr_loss=tr_res['loss'],
                tr_acc=tr_res['acc'],
                va_acc=va_res['acc'],
                va_auc=va_res['auc'],
                gate_w=va_res['gate_w']
            ))

            # Early stopping
            if va_res['auc'] > best_val_auc:
                best_val_auc = va_res['auc']
                patience = 0
                best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            else:
                patience += 1

            # Print progress
            if (epoch + 1) % 10 == 0 or epoch == 0:
                gw = va_res['gate_w']
                print(f"  Ep {epoch+1:3d}/{config['epochs']}: "
                      f"tr_acc={tr_res['acc']:.4f}, "
                      f"va_acc={va_res['acc']:.4f}, "
                      f"va_auc={va_res['auc']:.4f}, "
                      f"pat={patience}/{config['patience']}, "
                      f"GW=[f:{gw[0]:.2f}, m:{gw[1]:.2f}, c:{gw[2]:.2f}]")

            # Early stop
            if patience >= config['patience'] and epoch >= config['min_epochs']:
                print(f"✓ Early stopping at epoch {epoch + 1}")
                break

            if str(device) == 'cuda':
                torch.cuda.empty_cache()


In [ ]:
# ========================================================================
    # FINAL RESULTS
    # ========================================================================
    if all_results:
        print(f"\n{'='*80}")
        print(f"📈 FINAL {n_splits}-FOLD CROSS-VALIDATION RESULTS")
        print(f"{'='*80}\n")

        results_df = pd.DataFrame(all_results)

        print("Per-Fold Results:")
        print("-" * 80)
        for _, row in results_df.iterrows():
            print(f"Fold {int(row['fold'])}: "
                  f"Acc={row['accuracy']:.4f}  "
                  f"Sens={row['sensitivity']:.4f}  "
                  f"Spec={row['specificity']:.4f}  "
                  f"F1={row['f1_score']:.4f}  "
                  f"AUC={row['auc']:.4f}")

        print(f"\n{'AGGREGATED RESULTS':^80}")
        print(f"{'-'*80}")

        metrics_list = ['accuracy', 'sensitivity', 'specificity', 'precision', 'f1_score', 'auc']

        for metric in metrics_list:
            values = results_df[metric].values
            mean_val = np.mean(values)
            std_val = np.std(values)
            min_val = np.min(values)
            max_val = np.max(values)
            print(f"{metric.capitalize():12s}: {mean_val:.4f} ± {std_val:.4f} "
                  f"[min: {min_val:.4f}, max: {max_val:.4f}]")

        mean_acc = np.mean(results_df['accuracy'].values)
        std_acc = np.std(results_df['accuracy'].values)

        print(f"\n{'='*80}")
        print(f"🎯 FINAL ACCURACY: {mean_acc:.2%} ± {std_acc:.2%}")
        print(f"{'='*80}\n")

        return results_df, all_logs, data_dict

    return None, None, data_dict


In [ ]:
# ============================================================================
# 9. NILEARN ATLAS VISUALIZATION - 300 DPI
# ============================================================================

class BrainAtlasVisualizer:
    """Brain Atlas and Activation Visualization using nilearn"""
    
    def __init__(self, data_dict, fold_details, results_df, save_dir):
        self.data_dict = data_dict
        self.fold_details = fold_details
        self.results_df = results_df
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)
    
    def create_saliency_atlas(self):
        """Create saliency-based atlas visualization"""
        print("🧠 Creating saliency-based brain atlas...")
        
        # Compute mean saliency across all subjects
        conn_matrices = self.data_dict['connectivity']
        labels = self.data_dict['labels']
        
        asd_conn = conn_matrices[labels == 1]
        td_conn = conn_matrices[labels == 0]
        
        mean_asd = np.mean(asd_conn, axis=0)
        mean_td = np.mean(td_conn, axis=0)
        
        # Compute node importance
        asd_importance = np.sum(mean_asd, axis=1)
        td_importance = np.sum(mean_td, axis=1)
        
        # Normalize
        asd_importance = asd_importance / (np.max(asd_importance) + 1e-8)
        td_importance = td_importance / (np.max(td_importance) + 1e-8)
        
        # Create 4D array for atlas (treating each ROI as a "component")
        # Shape: (91, 109, 91, n_components) for MNI space
        affine = np.eye(4)
        n_rois = 200
        
        # Create simple atlas data - ROI importance as components
        atlas_data_asd = np.zeros((91, 109, 91, n_rois))
        atlas_data_td = np.zeros((91, 109, 91, n_rois))
        
        for i in range(n_rois):
            # Create simple spherical maps for each ROI
            atlas_data_asd[45, 54, 45, i] = asd_importance[i]
            atlas_data_td[45, 54, 45, i] = td_importance[i]
        
        # Create nibabel images
        atlas_img_asd = nib.Nifti1Image(atlas_data_asd, affine)
        atlas_img_td = nib.Nifti1Image(atlas_data_td, affine)
        
        # Plot using nilearn
        fig = plt.figure(figsize=(20, 10))
        gs = GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.3,
                     left=0.08, right=0.95, top=0.90, bottom=0.08)
        
        # ================================================================
        # Plot 1: ASD Atlas
        # ================================================================
        ax1 = fig.add_subplot(gs[0, :])
        try:
            plot_prob_atlas(atlas_img_asd, title="ASD Group - Brain Components Atlas")
            plt.savefig(os.path.join(self.save_dir, 'asd_atlas.png'), 
                       dpi=300, bbox_inches='tight', facecolor='white')
            plt.close()
            print("✅ Saved: asd_atlas.png (300 DPI)")
        except Exception as e:
            print(f"⚠️ Error plotting ASD atlas: {e}")
        
        # ================================================================
        # Plot 2: TD Atlas
        # ================================================================
        try:
            plot_prob_atlas(atlas_img_td, title="TD Group - Brain Components Atlas")
            plt.savefig(os.path.join(self.save_dir, 'td_atlas.png'), 
                       dpi=300, bbox_inches='tight', facecolor='white')
            plt.close()
            print("✅ Saved: td_atlas.png (300 DPI)")
        except Exception as e:
            print(f"⚠️ Error plotting TD atlas: {e}")
        
        return atlas_img_asd, atlas_img_td
    
    def plot_component_heatmaps(self):
        """Plot component heatmaps for all subjects"""
        print("📊 Generating component heatmaps...")
        
        conn_matrices = self.data_dict['connectivity']
        labels = self.data_dict['labels']
        
        asd_conn = conn_matrices[labels == 1]
        td_conn = conn_matrices[labels == 0]
        
        # Select top components by variance
        n_components = min(20, asd_conn.shape[0])
        
        fig = plt.figure(figsize=(22, 12))
        gs = GridSpec(3, 3, figure=fig, hspace=0.40, wspace=0.35,
                     left=0.07, right=0.96, top=0.90, bottom=0.08)
        
        # ================================================================
        # Plot 1: ASD mean connectivity
        # ================================================================
        ax1 = fig.add_subplot(gs[0, 0])
        mean_asd = np.mean(asd_conn, axis=0)
        im1 = ax1.imshow(mean_asd, cmap='hot', aspect='auto')
        ax1.set_title('ASD Mean Connectivity', fontsize=12, fontweight='bold')
        ax1.set_xlabel('ROI')
        ax1.set_ylabel('ROI')
        plt.colorbar(im1, ax=ax1, label='Correlation')
        
        # ================================================================
        # Plot 2: TD mean connectivity
        # ================================================================
        ax2 = fig.add_subplot(gs[0, 1])
        mean_td = np.mean(td_conn, axis=0)
        im2 = ax2.imshow(mean_td, cmap='hot', aspect='auto')
        ax2.set_title('TD Mean Connectivity', fontsize=12, fontweight='bold')
        ax2.set_xlabel('ROI')
        ax2.set_ylabel('ROI')
        plt.colorbar(im2, ax=ax2, label='Correlation')
        
        # ================================================================
        # Plot 3: Difference
        # ================================================================
        ax3 = fig.add_subplot(gs[0, 2])
        diff = mean_asd - mean_td
        im3 = ax3.imshow(diff, cmap='RdBu_r', aspect='auto', vmin=-0.3, vmax=0.3)
        ax3.set_title('Difference (ASD-TD)', fontsize=12, fontweight='bold')
        ax3.set_xlabel('ROI')
        ax3.set_ylabel('ROI')
        plt.colorbar(im3, ax=ax3, label='Δ Correlation')
        
        # ================================================================
        # Plot 4: ASD component variance
        # ================================================================
        ax4 = fig.add_subplot(gs[1, :2])
        asd_var = np.var(asd_conn, axis=0).sum(axis=1)
        top_asd_idx = np.argsort(asd_var)[-n_components:]
        ax4.bar(range(len(asd_var[top_asd_idx])), asd_var[top_asd_idx], 
               color='orangered', alpha=0.7, edgecolor='black')
        ax4.set_xlabel('Component Index', fontsize=11, fontweight='bold')
        ax4.set_ylabel('Variance', fontsize=11, fontweight='bold')
        ax4.set_title('ASD: Top Component Variances', fontsize=12, fontweight='bold')
        ax4.grid(True, alpha=0.3, axis='y')
        
        # ================================================================
        # Plot 5: TD component variance
        # ================================================================
        ax5 = fig.add_subplot(gs[1, 2])
        td_var = np.var(td_conn, axis=0).sum(axis=1)
        top_td_idx = np.argsort(td_var)[-n_components:]
        ax5.bar(range(len(td_var[top_td_idx])), td_var[top_td_idx], 
               color='dodgerblue', alpha=0.7, edgecolor='black')
        ax5.set_xlabel('Component Index', fontsize=11, fontweight='bold')
        ax5.set_ylabel('Variance', fontsize=11, fontweight='bold')
        ax5.set_title('TD: Top Component Variances', fontsize=12, fontweight='bold')
        ax5.grid(True, alpha=0.3, axis='y')
        
        # ================================================================
        # Plot 6: PCA components visualization
        # ================================================================
        ax6 = fig.add_subplot(gs[2, :])
        
        # Compute PCA-like components using SVD
        U_asd, S_asd, _ = np.linalg.svd(mean_asd, full_matrices=False)
        U_td, S_td, _ = np.linalg.svd(mean_td, full_matrices=False)
        
        n_plot = 10
        component_data = np.vstack([S_asd[:n_plot], S_td[:n_plot]])
        
        x = np.arange(n_plot)
        width = 0.35
        
        ax6.bar(x - width/2, S_asd[:n_plot] / np.sum(S_asd[:n_plot]), 
               width, label='ASD', color='orangered', alpha=0.8, edgecolor='black')
        ax6.bar(x + width/2, S_td[:n_plot] / np.sum(S_td[:n_plot]), 
               width, label='TD', color='dodgerblue', alpha=0.8, edgecolor='black')
        
        ax6.set_xlabel('Component Number', fontsize=12, fontweight='bold')
        ax6.set_ylabel('Normalized Variance', fontsize=12, fontweight='bold')
        ax6.set_title('Top 10 Principal Components Comparison', fontsize=13, fontweight='bold')
        ax6.set_xticks(x)
        ax6.set_xticklabels([f'PC{i+1}' for i in range(n_plot)])
        ax6.legend(fontsize=11)
        ax6.grid(True, alpha=0.3, axis='y')
        
        fig.suptitle('Brain Component Analysis using nilearn-style Visualization\n' +
                    'Component Extraction from Connectivity Data',
                    fontsize=18, fontweight='bold', y=0.96)
        
        save_path = os.path.join(self.save_dir, 'component_heatmaps.png')
        plt.savefig(save_path, dpi=300, bbox_inches='tight', 
                   facecolor='white', pad_inches=0.3)
        print(f"✅ Saved: {save_path} (300 DPI)")
        plt.close()
    
    def plot_network_components(self):
        """Plot network components with nilearn-style visualization"""
        print("🧠 Plotting network components...")
        
        conn_matrices = self.data_dict['connectivity']
        labels = self.data_dict['labels']
        
        asd_conn = conn_matrices[labels == 1]
        td_conn = conn_matrices[labels == 0]
        
        mean_asd = np.mean(asd_conn, axis=0)
        mean_td = np.mean(td_conn, axis=0)
        
        # Define networks
        networks = {
            'DMN': list(range(0, 30)),
            'SN': list(range(30, 60)),
            'ECN': list(range(60, 90)),
            'VN': list(range(90, 120)),
            'SMN': list(range(120, 150)),
        }
        
        fig = plt.figure(figsize=(22, 14))
        gs = GridSpec(3, len(networks), figure=fig, hspace=0.35, wspace=0.30,
                     left=0.05, right=0.98, top=0.90, bottom=0.08)
        
        net_names = list(networks.keys())
        
        for idx, net_name in enumerate(net_names):
            roi_list = networks[net_name]
            

In [ ]:
# ================================================================
            # ASD network component
            # ================================================================
            ax1 = fig.add_subplot(gs[0, idx])
            asd_net = mean_asd[np.ix_(roi_list, roi_list)]
            im1 = ax1.imshow(asd_net, cmap='hot', aspect='auto')
            ax1.set_title(f'ASD {net_name}', fontsize=11, fontweight='bold')
            ax1.set_xlabel('ROI', fontsize=10)
            ax1.set_ylabel('ROI', fontsize=10)
            plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
            
            # ================================================================
            # TD network component
            # ================================================================
            ax2 = fig.add_subplot(gs[1, idx])
            td_net = mean_td[np.ix_(roi_list, roi_list)]
            im2 = ax2.imshow(td_net, cmap='hot', aspect='auto')
            ax2.set_title(f'TD {net_name}', fontsize=11, fontweight='bold')
            ax2.set_xlabel('ROI', fontsize=10)
            ax2.set_ylabel('ROI', fontsize=10)
            plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
            
            # ================================================================
            # Network strength comparison
            # ================================================================
            ax3 = fig.add_subplot(gs[2, idx])
            
            asd_strength = np.mean(asd_net, axis=1)
            td_strength = np.mean(td_net, axis=1)
            
            x = np.arange(len(asd_strength))
            ax3.plot(x, asd_strength, marker='o', color='red', label='ASD', 
                    linewidth=2, markersize=5, alpha=0.7)
            ax3.plot(x, td_strength, marker='s', color='blue', label='TD', 
                    linewidth=2, markersize=5, alpha=0.7)
            
            ax3.set_xlabel('ROI', fontsize=10, fontweight='bold')
            ax3.set_ylabel('Mean Connectivity', fontsize=10, fontweight='bold')
            ax3.set_title(f'{net_name} Strength', fontsize=11, fontweight='bold')
            ax3.legend(fontsize=9)
            ax3.grid(True, alpha=0.3)
        
        fig.suptitle('Network-Specific Component Analysis\n' +
                    'Using nilearn-style Multi-component Visualization',
                    fontsize=18, fontweight='bold', y=0.96)
        
        save_path = os.path.join(self.save_dir, 'network_components.png')
        plt.savefig(save_path, dpi=300, bbox_inches='tight', 
                   facecolor='white', pad_inches=0.3)
        print(f"✅ Saved: {save_path} (300 DPI)")
        plt.close()
    
    def plot_component_loadings(self):
        """Plot component loadings like ICA components"""
        print("📊 Plotting component loadings...")
        
        conn_matrices = self.data_dict['connectivity']
        labels = self.data_dict['labels']
        
        asd_conn = conn_matrices[labels == 1]
        td_conn = conn_matrices[labels == 0]
        
        mean_asd = np.mean(asd_conn, axis=0)
        mean_td = np.mean(td_conn, axis=0)
        
        # Compute SVD
        U_asd, S_asd, Vt_asd = np.linalg.svd(mean_asd, full_matrices=False)
        U_td, S_td, Vt_td = np.linalg.svd(mean_td, full_matrices=False)
        
        n_components = min(9, U_asd.shape[1])
        
        fig = plt.figure(figsize=(22, 14))
        gs = GridSpec(3, 3, figure=fig, hspace=0.35, wspace=0.35,
                     left=0.08, right=0.96, top=0.90, bottom=0.08)
        

In [ ]:
 # ================================================================
        # ASD components
        # ================================================================
        for i in range(n_components):
            row = i // 3
            col = i % 3
            
            if row == 0:  # ASD row
                ax = fig.add_subplot(gs[0, col])
                component = U_asd[:, i].reshape(1, -1)
                im = ax.imshow(component, cmap='RdBu_r', aspect='auto', vmin=-0.3, vmax=0.3)
                ax.set_title(f'ASD Component {i+1}\n(Var: {S_asd[i]:.3f})', 
                           fontsize=11, fontweight='bold')
                ax.set_ylabel('Loading', fontsize=10)
                plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            
            elif row == 1:  # TD row
                ax = fig.add_subplot(gs[1, col])
                component = U_td[:, i].reshape(1, -1)
                im = ax.imshow(component, cmap='RdBu_r', aspect='auto', vmin=-0.3, vmax=0.3)
                ax.set_title(f'TD Component {i+1}\n(Var: {S_td[i]:.3f})', 
                           fontsize=11, fontweight='bold')
                ax.set_ylabel('Loading', fontsize=10)
                plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            
            else:  # Variance comparison
                ax = fig.add_subplot(gs[2, col])
                comps = np.arange(min(15, len(S_asd)))
                ax.plot(comps, S_asd[comps] / np.sum(S_asd), marker='o', 
                       color='red', label='ASD', linewidth=2, markersize=6)
                ax.plot(comps, S_td[comps] / np.sum(S_td), marker='s', 
                       color='blue', label='TD', linewidth=2, markersize=6)
                ax.set_xlabel('Component', fontsize=10, fontweight='bold')
                ax.set_ylabel('Explained Variance', fontsize=10, fontweight='bold')
                ax.set_title('Component Variance Ratio', fontsize=11, fontweight='bold')
                ax.legend(fontsize=9)
                ax.grid(True, alpha=0.3)
        
        fig.suptitle('Independent Component-like Analysis\n' +
                    'SVD-based Component Loadings (ICA-style)',
                    fontsize=18, fontweight='bold', y=0.96)
        
        save_path = os.path.join(self.save_dir, 'component_loadings.png')
        plt.savefig(save_path, dpi=300, bbox_inches='tight', 
                   facecolor='white', pad_inches=0.3)
        print(f"✅ Saved: {save_path} (300 DPI)")
        plt.close()
    
    def run_full_atlas_visualization(self):
        """Run complete atlas visualization"""
        print(f"\n{'='*80}")
        print("🧠 BRAIN ATLAS VISUALIZATION (NILEARN-STYLE)")
        print("   Output: 300 DPI High-Quality PNG Images")
        print(f"{'='*80}\n")
        
        print("📊 Generating visualizations...")
        self.create_saliency_atlas()
        self.plot_component_heatmaps()
        self.plot_network_components()
        self.plot_component_loadings()
        
        print(f"\n✅ Brain atlas visualization complete!")
        print(f"   All figures saved to: {self.save_dir}")



In [ ]:
# ============================================================================
# 10. MAIN TRAINING AND ANALYSIS PIPELINE
# ============================================================================

def main():
    """Main training and analysis pipeline"""
    print("="*80)
    print("🚀 HYBRID GCN-CNN NETWORK FOR ASD DIAGNOSIS (5-FOLD CV)")
    print("   WITH NILEARN-STYLE BRAIN ATLAS VISUALIZATION")
    print("="*80)

    set_seed(42)

    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n🔧 Device: {device}")
    if str(device) == 'cuda':
        print(f"   GPU: {torch.cuda.get_device_name(0)}")
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"   Memory: {gpu_mem:.2f} GB")

    # Data loading
    base_path = "/public/home/hpc204166/newHaider1/ABIDE_DATA"
    print(f"\n📁 Data path: {base_path}")

    if not os.path.exists(base_path):
        print(f"❌ Data path does not exist")
        return

    data_dict = load_all_data(base_path)

    if data_dict is None:
        print("❌ Failed to load data")
        return

    # Training config
    config = {
        'epochs': 100,
        'batch_size': 8,
        'lr': 3e-4,
        'dropout': 0.3,
        'hidden_dim': 128,
        'gcn_hidden': 128,
        'num_heads': 4,
        'patience': 25,
        'min_epochs': 20,
        'weight_decay': 1e-4,
        'focal_gamma': 2.0,
        'mixup_alpha': 0.2,
        'div_weight': 0.2,
        'balanced_sampler': True,
    }

    # Train
    results_df, fold_details, data_dict = train_model(data_dict, device, n_splits=5, config=config)

    # Save and analyze
    if results_df is not None:
        print(f"\n{'='*80}")
        print("🎉 TRAINING COMPLETE - SAVING RESULTS & RUNNING ANALYSIS")
        print(f"{'='*80}")

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        save_dir = f"asd_gcn_cnn_5fold_results_{timestamp}"
        os.makedirs(save_dir, exist_ok=True)

        # Save CSV and pickle
        csv_path = os.path.join(save_dir, 'cv_results.csv')
        results_df.to_csv(csv_path, index=False)
        print(f"✅ Results saved to: {csv_path}")

        pkl_path = os.path.join(save_dir, 'cv_results.pkl')
        with open(pkl_path, 'wb') as f:
            pickle.dump(results_df.to_dict(), f)

        fold_path = os.path.join(save_dir, 'fold_details.pkl')
        with open(fold_path, 'wb') as f:
            pickle.dump(fold_details, f)

        # ====================================================================
        # BRAIN ATLAS VISUALIZATION WITH NILEARN
        # ====================================================================
        print("\n" + "="*80)
        print("🧠 RUNNING BRAIN ATLAS VISUALIZATION (NILEARN-STYLE)")
        print("="*80)
        
        atlas_dir = os.path.join(save_dir, 'brain_atlas_visualization')
        atlas_viz = BrainAtlasVisualizer(data_dict, fold_details, results_df, atlas_dir)
        atlas_viz.run_full_atlas_visualization()

        # Save summary
        summary_path = os.path.join(save_dir, 'summary.txt')
        with open(summary_path, 'w') as f:
            f.write("="*80 + "\n")
            f.write("HYBRID GCN-CNN - ASD DIAGNOSIS RESULTS (5-FOLD CV)\n")
            f.write("WITH NILEARN-STYLE BRAIN ATLAS VISUALIZATION\n")
            f.write("="*80 + "\n\n")
            f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Total Dataset Size: {len(data_dict['connectivity'])}\n")
            f.write(f"Number of Folds: 5\n\n")

            f.write("Configuration:\n")
            f.write("-"*80 + "\n")
            for key, value in config.items():
                f.write(f"{key:20s}: {value}\n")
            f.write("\n")

            f.write("Per-Fold Results:\n")
            f.write("-"*80 + "\n")
            for _, row in results_df.iterrows():
                f.write(f"Fold {int(row['fold'])}: "
                       f"Acc={row['accuracy']:.4f}, "
                       f"Sens={row['sensitivity']:.4f}, "
                       f"Spec={row['specificity']:.4f}, "
                       f"F1={row['f1_score']:.4f}, "
                       f"AUC={row['auc']:.4f}\n")

            f.write("\nAggregated Metrics:\n")
            f.write("-"*80 + "\n")
            metrics_list = ['accuracy', 'sensitivity', 'specificity', 'precision', 'f1_score', 'auc']
            for metric in metrics_list:
                values = results_df[metric].values
                mean_val = np.mean(values)
                std_val = np.std(values)
                min_val = np.min(values)
                max_val = np.max(values)
                f.write(f"{metric.capitalize():12s}: {mean_val:.4f} ± {std_val:.4f} "
                       f"(min: {min_val:.4f}, max: {max_val:.4f})\n")

            mean_acc = np.mean(results_df['accuracy'].values)
            std_acc = np.std(results_df['accuracy'].values)
            f.write(f"\n{'='*80}\n")
            f.write(f"Final Accuracy: {mean_acc:.4f} ± {std_acc:.4f}\n")
            f.write(f"{'='*80}\n\n")

            f.write("BRAIN ATLAS VISUALIZATIONS (All 300 DPI):\n")
            f.write("-"*80 + "\n")
            f.write("1. Brain Component Atlas:\n")
            f.write("   - asd_atlas.png (ASD group atlas)\n")
            f.write("   - td_atlas.png (TD group atlas)\n\n")
            f.write("2. Component Heatmaps:\n")
            f.write("   - component_heatmaps.png (All components with variance)\n\n")
            f.write("3. Network Components:\n")
            f.write("   - network_components.png (DMN, SN, ECN, VN, SMN)\n\n")
            f.write("4. Component Loadings (ICA-style):\n")
            f.write("   - component_loadings.png (SVD-based ICA visualization)\n\n")

            f.write("KEY FEATURES:\n")
            f.write("-"*80 + "\n")
            f.write("✅ Nilearn-style probability atlas visualization\n")
            f.write("✅ Multi-component brain analysis\n")
            f.write("✅ Network-specific component extraction\n")
            f.write("✅ ICA-style component loadings\n")
            f.write("✅ SVD-based dimensionality reduction\n")
            f.write("✅ High-resolution 300 DPI outputs\n")

        print(f"✅ Summary saved to: {summary_path}")

        print(f"\n{'='*80}")
        print("🎊 ALL ANALYSIS COMPLETE!")
        print(f"{'='*80}")
        print(f"\n📊 Generated Outputs:")
        print(f"  • Brain Atlas Visualizations (300 DPI): 4 major figures")
        print(f"  • Component Heatmaps with variance analysis")
        print(f"  • Network-specific components (5 networks)")
        print(f"  • ICA-style component loadings")
        print(f"  • Results CSV and PKL files")
        print(f"  • Comprehensive summary report")
        print(f"\n📁 Location: {save_dir}/")
        print(f"\n✅ All outputs are publication-ready at 300 DPI!")
        print(f"{'='*80}\n")
    else:
        print("\n❌ Training failed to produce results")


if __name__ == "__main__":
    main()

In [ ]:
# ====================================================================
        # TEST
        # ====================================================================
        if best_state:
            model.load_state_dict(best_state)

        print(f"\n🧪 Testing (best_va_auc={best_val_auc:.4f})…")

        model.eval()
        test_preds = []
        test_probs = []
        test_labels_batch = []

        with torch.no_grad():
            for conn_batch, smri_batch, labels_batch in test_loader:
                conn_batch = conn_batch.to(device).float()
                smri_batch = smri_batch.to(device).float()
                labels_batch = labels_batch.to(device)

                try:
                    outputs, extras = model(conn_batch, smri_batch)
                    probs = torch.softmax(outputs, dim=1)
                    preds = torch.argmax(outputs, dim=1)

                    test_preds.extend(preds.cpu().numpy())
                    test_probs.extend(probs[:, 1].cpu().numpy())
                    test_labels_batch.extend(labels_batch.cpu().numpy())

                    del conn_batch, smri_batch, labels_batch, outputs, probs, preds
                except RuntimeError as e:
                    if "out of memory" in str(e):
                        torch.cuda.empty_cache()
                        continue
                    else:
                        raise

        test_preds = np.array(test_preds)
        test_probs = np.array(test_probs)
        test_labels = np.array(test_labels_batch)

        # Calculate metrics
        metrics = calc_metrics(test_labels, test_preds, test_probs)

        if metrics:
            print(f"\n{'='*80}")
            print(f"📊 FOLD {fold_idx + 1} TEST RESULTS")
            print(f"{'='*80}")
            print(f"   Accuracy:   {metrics['accuracy']:.4f}")
            print(f"   Sensitivity:{metrics['sensitivity']:.4f}")
            print(f"   Specificity:{metrics['specificity']:.4f}")
            print(f"   Precision:  {metrics['precision']:.4f}")
            print(f"   F1-Score:   {metrics['f1_score']:.4f}")
            print(f"   AUC:        {metrics['auc']:.4f}")
            print(f"\n   Confusion Matrix:")
            print(f"      TN={metrics['tn']:3d}  FP={metrics['fp']:3d}")
            print(f"      FN={metrics['fn']:3d}  TP={metrics['tp']:3d}")

            fold_result = {
                'fold': fold_idx + 1,
                **metrics,
                'best_val_auc': best_val_auc,
                'epochs_trained': len(epoch_log)
            }
            all_results.append(fold_result)

            all_logs.append({
                'fold': fold_idx + 1,
                'epochs': epoch_log,
                'final_metrics': fold_result,
                'test_predictions': {
                    'y_true': test_labels,
                    'y_probs': test_probs,
                    'y_pred': test_preds
                },
                'test_loader': test_loader,
                'model_state': best_state
            })
        else:
            print(f"⚠️ Failed to calculate metrics for fold {fold_idx + 1}")

        # Cleanup
        del model
        torch.cuda.empty_cache()
        gc.collect()
